In [ ]:
# Seed the eventstream raw bronze table from committed synthetic envelopes.
# Parameters (Fabric injects overrides via the papermill-compatible 'parameters' tag).
target_lakehouse = 'lh_ihzhhpf_sit'
seed_path = 'Files/eventstream-seed/eventstream_raw.json'   # lakehouse-relative committed corpus
raw_table = 'Tables/bronze_eventstream_raw'                 # path-based Delta the eventstream bronze notebook reads
run_id = 'run-manual-local'

# Local dry-run fallback (repo-relative) so CI can execute this cell without a
# Spark session and still exit 0 (guarded by NameError on `spark` below).
from pathlib import Path
REPO = Path.cwd().parents[2] if (Path.cwd().name == 'eventstream') else Path.cwd()
LOCAL_SEED = REPO / 'data' / 'synthetic' / 'eventstream' / 'eventstream_raw.json'
print(f'seed_path (lakehouse): {seed_path}')
print(f'raw_table           : {raw_table}')

In [ ]:
# --- Load the committed envelope corpus --------------------------------
# Fabric context: read the JSON from the lakehouse Files/ mount.
# Local dry-run: NameError on `spark` falls back to the repo-relative fixture.
import json

try:
    seed_text = '\n'.join(
        r.value for r in spark.read.option('multiline', 'true').text(seed_path).collect()  # type: ignore[name-defined]
    )
    doc = json.loads(seed_text)
except NameError:
    doc = json.loads(LOCAL_SEED.read_text(encoding='utf-8'))

assert doc['contractId'] == 'DC-EVENTSTREAM-RAW-v1', f"unexpected contract {doc.get('contractId')}"
records = doc['records']
kinds = {}
for r in records:
    kinds[r['eventKind']] = kinds.get(r['eventKind'], 0) + 1
print(f'Loaded {len(records)} eventstream envelope(s)')
for k in sorted(kinds):
    print(f'  {k:<28s} {kinds[k]}')

In [ ]:
# --- Materialise Tables/bronze_eventstream_raw (Fabric-context only) ----
# The eventstream bronze notebook (01_bronze_eventstream, batch mode) reads
# this path-based Delta table and routes envelopes per eventKind. We write an
# EXPLICIT schema (payload stays a JSON string; silver Gate 1 accepts strings
# and gold _flatten_payload reads them via get_json_object) and OVERWRITE so
# the seed is idempotent across reruns.
try:
    from pyspark.sql.types import StructType, StructField, StringType, LongType
    schema = StructType([
        StructField('eventKind', StringType(), False),
        StructField('eventId', StringType(), False),
        StructField('hospitalId', StringType(), False),
        StructField('simulatedAt', StringType(), True),
        StructField('emittedAt', StringType(), True),
        StructField('simRunId', StringType(), True),
        StructField('seed', LongType(), True),
        StructField('payload', StringType(), True),
    ])
    rows = [(
        r['eventKind'], r['eventId'], r['hospitalId'], r.get('simulatedAt'),
        r.get('emittedAt'), r.get('simRunId'), int(r.get('seed', 0)), r.get('payload'),
    ) for r in records]
    df = spark.createDataFrame(rows, schema=schema)  # type: ignore[name-defined]
    (df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').save(raw_table))
    print(f'Wrote {df.count()} envelope(s) -> {raw_table}')
except NameError:
    print('LOCAL DRY-RUN: `spark` not bound. Skipping Delta write.')
    print(f'  would write {len(records)} envelope(s) -> {raw_table}')